In [7]:
import requests
import json
import re
import pandas as pd

def http_function(endpoint, **http_args):
    try:
        resp = requests.get(endpoint, **http_args)
        return resp
    except requests.exceptions.RequestException as e:
        class ErrorResponse:
            def __init__(self, error):
                self.status_code = 500
                self._error = error
            def json(self):
                return {"error": str(self._error)}
            def raise_for_status(self):
                raise requests.exceptions.RequestException(self._error)
        return ErrorResponse(e)

def get_uniprot(accession):
    endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
    headers = {"Accept": "application/json"}
    return http_function(endpoint, headers=headers)

def uniprot_parse_response(resp):
    try:
        if resp.status_code != 200:
            try:
                error_data = resp.json()
                if "messages" in error_data:
                    return {"error": error_data["messages"][0]}
                else:
                    return {"error": f"HTTP Error {resp.status_code}"}
            except:
                return {"error": f"HTTP Error {resp.status_code}"}
        
        data = resp.json()
        accession = data.get("primaryAccession")
        
        organism = data.get("organism", {})
        organism_name = organism.get("scientificName") if organism else None
        
        genes = data.get("genes", [])
        gene_info = []
        for gene in genes:
            gene_entry = {}
            if "geneName" in gene:
                gene_entry["geneName"] = gene["geneName"]
            if "synonyms" in gene:
                gene_entry["synonyms"] = gene["synonyms"]
            if gene_entry:
                gene_info.append(gene_entry)
        
        sequence = data.get("sequence", {})
        sequence_info = {
            "value": sequence.get("value", ""),
            "length": sequence.get("length", 0),
            "molWeight": sequence.get("molWeight", 0),
            "crc64": sequence.get("crc64", ""),
            "md5": sequence.get("md5", "")
        } if sequence else None
        
        protein_type = "protein"
        if "proteinDescription" in data:
            prot_desc = data["proteinDescription"]
            if "recommendedName" in prot_desc and "fullName" in prot_desc["recommendedName"]:
                protein_type = prot_desc["recommendedName"]["fullName"].get("value", "protein")
            elif "submittedName" in prot_desc and len(prot_desc["submittedName"]) > 0:
                protein_type = prot_desc["submittedName"][0].get("fullName", {}).get("value", "protein")
        
        output = {
            accession: {
                "organism": organism_name,
                "geneInfo": gene_info if gene_info else None,
                "sequenceInfo": sequence_info,
                "type": protein_type
            }
        }
        return output
        
    except Exception as e:
        return {"error": str(e)}

def get_ensembl(id):
    endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
    headers = {"Content-Type": "application/json"}
    params = {"expand": 1}
    return http_function(endpoint, headers=headers, params=params)

def ensembl_parse_response(resp):
    try:
        if resp.status_code != 200:
            try:
                error_data = resp.json()
                if "error" in error_data:
                    return {"error": error_data["error"]}
                else:
                    return {"error": f"HTTP Error {resp.status_code}"}
            except:
                return {"error": f"HTTP Error {resp.status_code}"}
        
        data = resp.json()
        ensembl_id = data.get("id")
        
        output = {
            ensembl_id: {
                "object_type": data.get("object_type"),
                "assembly_name": data.get("assembly_name"),
                "species": data.get("species"),
                "db_type": data.get("db_type"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "id": data.get("id"),
                "description": data.get("description"),
                "canonical_transcript": data.get("canonical_transcript"),
                "source": data.get("source")
            }
        }
        
        return output
        
    except Exception as e:
        return {"error": str(e)}

def identify_database(id):
    uniprot_patterns = [
        r'^[OPQ][0-9][A-Z0-9]{3}[0-9]$',
        r'^[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]{0,3}$',
        r'^[A-NR-Z][0-9]{5}$',
    ]
    
    ensembl_patterns = [
        r'^ENS[A-Z]*G\d{11}$',
        r'^ENS[A-Z]*T\d{11}$',
        r'^ENS[A-Z]*P\d{11}$',
        r'^ENS[A-Z]*E\d{11}$',
    ]
    
    for pattern in uniprot_patterns:
        if re.match(pattern, id):
            return "uniprot"
    
    for pattern in ensembl_patterns:
        if re.match(pattern, id):
            return "ensembl"
    
    return "unknown"

def main(ids):
    output = {}
    
    for id in ids:
        id = id.strip()
        db_type = identify_database(id)
        
        try:
            if db_type == "uniprot":
                resp = get_uniprot(id)
                parsed = uniprot_parse_response(resp)
                
                if "error" in parsed:
                    output[id] = parsed["error"]
                else:
                    for key, value in parsed.items():
                        output[key] = value
                        
            elif db_type == "ensembl":
                resp = get_ensembl(id)
                parsed = ensembl_parse_response(resp)
                
                if "error" in parsed:
                    output[id] = parsed["error"]
                else:
                    for key, value in parsed.items():
                        output[key] = value
                        
            else:
                output[id] = "error:unknown database"
                
        except Exception as e:
            output[id] = f"error:{str(e)}"
    
    return output

def create_dataframe_from_output(output):
    rows = []
    
    for id_key, value in output.items():
        if isinstance(value, dict):
            row = {"ID": id_key}
            for k, v in value.items():
                if isinstance(v, (dict, list)):
                    row[k] = json.dumps(v, ensure_ascii=False, default=str)
                else:
                    row[k] = v
            rows.append(row)
        else:
            rows.append({"ID": id_key, "error": value})
    
    df = pd.DataFrame(rows)
    if "ID" in df.columns:
        df.set_index("ID", inplace=True)
    return df

if __name__ == "__main__":
    try:
        with open("test_data.txt") as f:
            ids = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    except FileNotFoundError:
        ids = ["P11473", "Q91XI3", "hello", "ENSG00000157764", "ENSG00000139618"]
        with open("test_data.txt", "w") as f:
            for id in ids:
                f.write(f"{id}\n")
        print("Created test_data.txt with test IDs")
    
    print("Input IDs:", ids)
    print("-" * 80)
    
    output_dict = main(ids)
    
    for id_key, value in output_dict.items():
        print(f"\n{id_key}:")
        if isinstance(value, dict):
            print(json.dumps(value, indent=2, ensure_ascii=False)[:500] + "...")
        else:
            print(f"  {value}")
    
    df_result = create_dataframe_from_output(output_dict)
    print("\n" + "="*80)
    print("DataFrame:")
    print("="*80)
    print(df_result)
    
    df_result.to_csv("hw2_res.csv")
    print("\nResults were saved to hw2_results.csv")

Input IDs: ['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618']
--------------------------------------------------------------------------------

P11473:
{
  "organism": "Homo sapiens",
  "geneInfo": [
    {
      "geneName": {
        "evidences": [
          {
            "evidenceCode": "ECO:0000312",
            "source": "HGNC",
            "id": "HGNC:12679"
          }
        ],
        "value": "VDR"
      },
      "synonyms": [
        {
          "value": "NR1I1"
        }
      ]
    }
  ],
  "sequenceInfo": {
    "value": "MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKE...

Q91XI3:
{
  "organism": "Ictidomys tridecemlineatus",
  "geneInfo": [
    {
      "geneName": {
        "value": "INS"
      }
    }
  ],
  "sequenceInfo": {
    "value": "MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVEALYLVCGERGFFYTPKSRREVEEQQGGQVELGGGPGAGLPQPLALEMALQKRGIVEQCCTSICSLYQLENYCN",
    "length": 110,
    "molWeight": 12004